# 📊 Customer Churn Prediction — End-to-End Data Science Project
**By: Allu Hema Srivalli**  
*IBM Python for Data Science Certificate Project*

---

## 🎯 Problem Statement
A telecom company is losing customers (churning). Our goal is to:
1. **Understand** why customers leave using Exploratory Data Analysis (EDA)
2. **Predict** which customers are likely to churn using Machine Learning
3. **Recommend** business actions to reduce churn

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)


## 📦 Section 1: Import Libraries & Load Data

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve)

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('Set2')
sns.set_style('whitegrid')

print('All libraries imported successfully!')

In [ ]:
# Load dataset
df = pd.read_csv('Telco-Customer-Churn.csv')

print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nFirst 5 rows:')
df.head()

## 🔍 Section 2: Data Understanding & Cleaning

In [ ]:
# Basic info
print('=== DATASET INFO ===')
print(f'Total Customers   : {len(df)}')
print(f'Total Features    : {df.shape[1]}')
churn_rate = round(df['Churn'].value_counts(normalize=True)['Yes']*100, 2)
print(f'Churn Rate        : {churn_rate}%')
print('\n=== DATA TYPES ===')
print(df.dtypes)

In [ ]:
# Check and fix data quality issues
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

# TotalCharges has spaces — fix it
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID (not useful for analysis)
df.drop('customerID', axis=1, inplace=True)

print('\nData cleaning complete!')
print(f'Final shape: {df.shape}')

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df.describe().round(2)

## 📈 Section 3: Exploratory Data Analysis (EDA)

### 3.1 — Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=2, width=0.5)
axes[0].set_title('Customer Churn Count', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=13)

axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold', pad=15)

plt.suptitle('Overall Churn Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nInsight: ~26.5% customers churned — this is a significant business problem!')

### 3.2 — Contract Type & Tenure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100).reset_index()
contract_churn.columns = ['Contract', 'Churn_Rate']
contract_churn = contract_churn.sort_values('Churn_Rate', ascending=False)

bars = axes[0].bar(contract_churn['Contract'], contract_churn['Churn_Rate'],
                    color=['#e74c3c','#f39c12','#2ecc71'], edgecolor='white', linewidth=2, width=0.5)
axes[0].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 60)
for bar, val in zip(bars, contract_churn['Churn_Rate']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

churned = df[df['Churn'] == 'Yes']['tenure']
retained = df[df['Churn'] == 'No']['tenure']
axes[1].hist(retained, bins=30, alpha=0.7, color='#2ecc71', label='Retained', edgecolor='white')
axes[1].hist(churned, bins=30, alpha=0.7, color='#e74c3c', label='Churned', edgecolor='white')
axes[1].set_title('Tenure Distribution by Churn', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Tenure (Months)')
axes[1].set_ylabel('Number of Customers')
axes[1].legend(fontsize=11)

plt.suptitle('Contract & Tenure Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('contract_tenure_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('Insight: Month-to-month customers churn at 42.7% vs only 2.8% for 2-year contracts!')
print('Insight: New customers (low tenure) are most at risk of churning!')

### 3.3 — Monthly Charges & Internet Service

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

churn_no = df[df['Churn'] == 'No']['MonthlyCharges']
churn_yes = df[df['Churn'] == 'Yes']['MonthlyCharges']
axes[0].boxplot([churn_no, churn_yes], labels=['No Churn', 'Churned'],
               patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Monthly Charges vs Churn', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Monthly Charges ($)')

internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100).reset_index()
internet_churn.columns = ['InternetService', 'Churn_Rate']
colors_int = ['#e74c3c','#f39c12','#2ecc71']
bars2 = axes[1].bar(internet_churn['InternetService'], internet_churn['Churn_Rate'],
                     color=colors_int, edgecolor='white', linewidth=2, width=0.5)
axes[1].set_title('Churn Rate by Internet Service', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Internet Service Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 50)
for bar, val in zip(bars2, internet_churn['Churn_Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('charges_internet_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('Insight: Churned customers pay more monthly (~$74 vs ~$61)!')
print('Insight: Fiber optic users churn at 41.9% — possibly due to high cost!')

### 3.4 — Feature Correlation with Churn

In [ ]:
df_corr = df.copy()
le = LabelEncoder()
for col in df_corr.select_dtypes(include='object').columns:
    df_corr[col] = le.fit_transform(df_corr[col])

churn_corr = df_corr.corr()['Churn'].drop('Churn').sort_values(ascending=False)

plt.figure(figsize=(10, 7))
colors_corr = ['#e74c3c' if x > 0 else '#2ecc71' for x in churn_corr.values]
plt.barh(churn_corr.index, churn_corr.values, color=colors_corr, edgecolor='white', linewidth=1.5)
plt.axvline(x=0, color='black', linewidth=1, linestyle='--', alpha=0.5)
plt.title('Feature Correlation with Churn', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Correlation Coefficient')
red_patch = mpatches.Patch(color='#e74c3c', label='Positive (increases churn risk)')
green_patch = mpatches.Patch(color='#2ecc71', label='Negative (reduces churn risk)')
plt.legend(handles=[red_patch, green_patch], loc='lower right')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Top factors INCREASING churn: Month-to-month contract, Fiber optic, Paperless billing')
print('Top factors REDUCING churn: Long tenure, 2-year contract, Online security')

## 🤖 Section 4: Machine Learning — Churn Prediction

### 4.1 — Preprocessing

In [ ]:
df_ml = df.copy()
le = LabelEncoder()

categorical_cols = df_ml.select_dtypes(include='object').columns.tolist()
print(f'Encoding {len(categorical_cols)} categorical columns...')
for col in categorical_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set  : {X_train.shape[0]} samples')
print(f'Testing set   : {X_test.shape[0]} samples')

### 4.2 — Train & Evaluate Models

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

# Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)
dt_proba = dt_model.predict_proba(X_test_scaled)[:, 1]

print('=' * 50)
print(f'{"MODEL":<25} {"ACCURACY":<12} {"ROC-AUC"}')
print('=' * 50)
lr_acc = accuracy_score(y_test, lr_pred)*100
dt_acc = accuracy_score(y_test, dt_pred)*100
lr_auc = roc_auc_score(y_test, lr_proba)
dt_auc = roc_auc_score(y_test, dt_proba)
print(f'{"Logistic Regression":<25} {lr_acc:.2f}%      {lr_auc:.4f}')
print(f'{"Decision Tree":<25} {dt_acc:.2f}%      {dt_auc:.4f}')
print('=' * 50)
print('\nDetailed Report — Logistic Regression:')
print(classification_report(y_test, lr_pred, target_names=['No Churn', 'Churned']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm_lr = confusion_matrix(y_test, lr_pred)
im = axes[0].imshow(cm_lr, interpolation='nearest', cmap='Blues')
axes[0].set_title(f'Confusion Matrix — Logistic Regression\nAccuracy: {lr_acc:.1f}%',
                   fontsize=13, fontweight='bold', pad=15)
axes[0].set_xticks([0,1])
axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No Churn','Churned'])
axes[0].set_yticklabels(['No Churn','Churned'])
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')
for i in range(2):
    for j in range(2):
        color = 'white' if cm_lr[i,j] > cm_lr.max()/2 else 'black'
        axes[0].text(j, i, str(cm_lr[i,j]), ha='center', va='center',
                     fontsize=24, color=color, fontweight='bold')

# ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba)
axes[1].plot(fpr_lr, tpr_lr, color='#3498db', linewidth=2.5,
             label=f'Logistic Regression (AUC = {lr_auc:.3f})')
axes[1].plot(fpr_dt, tpr_dt, color='#e74c3c', linewidth=2.5,
             label=f'Decision Tree (AUC = {dt_auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4, label='Random Classifier')
axes[1].fill_between(fpr_lr, tpr_lr, alpha=0.1, color='#3498db')
axes[1].set_title('ROC Curve Comparison', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].set_xlim([0,1])
axes[1].set_ylim([0,1])

plt.suptitle('Model Evaluation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(12)

plt.figure(figsize=(10, 7))
colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(feature_importance)))
plt.barh(feature_importance['Feature'], feature_importance['Importance'],
         color=colors_fi, edgecolor='white', linewidth=1.5)
plt.title('Top 12 Most Important Features for Churn Prediction',
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('Insight: Tenure, Monthly Charges, and Total Charges are top predictors!')

## 💼 Section 5: Business Insights & Recommendations

In [ ]:
insights = """
==============================================================
           BUSINESS INSIGHTS SUMMARY
==============================================================

1. CONTRACT TYPE IS #1 CHURN DRIVER
   Month-to-month: 42.7% churn rate
   Two-year contract: only 2.8% churn rate
   Action: Offer discounts to switch to annual plans

2. NEW CUSTOMERS ARE MOST AT RISK
   Customers in first 0-12 months churn most
   Long-tenure customers (50+ months) rarely leave
   Action: Create loyalty rewards / onboarding program

3. FIBER OPTIC USERS CHURN AT 41.9%
   Likely due to high cost or strong competition
   Action: Improve service quality / offer bundle discounts

4. HIGH MONTHLY CHARGES CORRELATE WITH CHURN
   Churned customers pay ~$74/mo vs $61 (retained)
   Action: Introduce flexible tiered pricing

==============================================================
MODEL PERFORMANCE
  Logistic Regression Accuracy : ~80%
  ROC-AUC Score                : ~0.85
==============================================================
"""
print(insights)

## ✅ Conclusion

This project demonstrates a complete end-to-end data science pipeline:

| Step | Tools Used | Outcome |
|------|------------|----------|
| Data Loading | `pandas` | Loaded 7,043 customer records |
| Data Cleaning | `pandas`, `numpy` | Fixed missing values, type errors |
| EDA | `matplotlib`, `seaborn` | Uncovered 4 key business insights |
| ML Modeling | `scikit-learn` | Logistic Regression + Decision Tree |
| Evaluation | `sklearn.metrics` | ~80% accuracy, AUC ~0.85 |

**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn  
**Certificate:** IBM Python for Data Science — Issued May 2026  
**Author:** Allu Hema Srivalli
